# 20_model_eval — 모델 학습 & 성능 평가

**한 줄 요약:** fingerprint만 / descriptor만 / 결합 세 입력을 각각 학습시켜 **점수로 비교**한다.
**용어:** 모델=데이터에서 규칙을 배워 정답을 맞히는 것(여기선 RandomForest) / 교차검증=데이터를 5조각 내어 4개로 배우고 1개로 시험하기를 5번.
**큰 흐름:** ① 도구 → ② 데이터·3입력 → ③ 채점 함수 → ④ 3입력 학습·채점 → ⑤ 그래프
> ⚠️ 지금 inactive 대부분이 decoy(구조가 일부러 다름)라 점수가 매우 높게(≈0.99) 나오는데, '실력'이 아니라 '문제가 쉬워서'다.

> **📌 이 노트북 읽는 법 (처음이면 여기부터)**
> - **셀(cell)** = 코드 한 덩어리. 위에서부터 하나씩 **실행**(`Shift`+`Enter`)한다. 앞 셀에서 만든 값을 뒤 셀이 쓰므로 **순서대로**.
> - 각 코드 셀은 **[① 무슨 작업인지 설명] → [② 코드] → [③ 🔎 코드 뜯어보기]** 순서로 놓았다.
>   ③은 그 셀에 **처음 나온** 함수·문법을 잘게 푼 것(이미 나온 건 반복 안 함).
> - 코드 줄 뒤 `# ...` 은 **주석**(설명)이라 실행에 영향 없음.

### 셀 1 — 도구 불러오기
모델·교차검증·그래프·점수 계산 도구를 가져온다.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             matthews_corrcoef, roc_curve, precision_recall_curve)

🔎 **코드 뜯어보기 (셀 1)** *(import/as/from은 18과 동일)*
- `sklearn` : **scikit-learn** — 파이썬 대표 머신러닝 라이브러리(모델·평가 도구가 다 들어있음).
- `RandomForestClassifier` : 우리가 쓸 **모델**. 결정트리 여러 개를 만들어 **다수결 투표**로 예측.
- `StratifiedKFold` : 데이터를 **클래스 비율을 유지한 채** 5조각으로 나누는 도구.
- `cross_val_predict` : 교차검증으로 각 샘플의 **'안 본 상태' 예측**을 구해주는 함수.
- `import matplotlib` + `matplotlib.use('Agg')` : 그래프 라이브러리. **Agg**는 화면 없이 **파일로 저장**하는 모드.
- `import matplotlib.pyplot as plt` : 그래프 그리는 부분을 **plt**로 부르기.
- `from sklearn.metrics import (...)` : 괄호로 **여러 개를 한 번에** 가져오기(roc_auc_score 등 점수 함수들).

### 셀 2 — 데이터 읽고 '입력 3종류' 만들기
정답 y와, fingerprint/descriptor/결합 세 입력을 준비한다.

In [ ]:
# 최종 학습데이터 로드 + 3가지 특징 집합 정의
SRC = 'data/HSD17B13_final_training_1to1.csv'
df = pd.read_csv(SRC)
y = df['potency'].to_numpy()

fp_cols = [c for c in df.columns if c.startswith('fp_')]
non_desc = set(['canonical_smiles', 'potency'] + fp_cols)
desc_cols = [c for c in df.columns if c not in non_desc]
print('화합물', len(df), '| potency', dict(pd.Series(y).value_counts()))
print('fingerprint 열', len(fp_cols), '| descriptor 열', len(desc_cols))

FEATURES = {
    'fingerprint':      df[fp_cols].to_numpy(),
    'descriptor':       df[desc_cols].to_numpy(),
    '결합(FP+desc)':    df[fp_cols + desc_cols].to_numpy(),
}

🔎 **코드 뜯어보기 (셀 2)**
- `.to_numpy()` : 표(pandas)를 모델이 먹기 좋은 **순수 숫자 배열(numpy)** 로 바꿈.
- `c.startswith('fp_')` : 글자가 'fp_'로 **시작하는지** True/False. → fingerprint 열만 골라내는 조건.
- `set([...])` : **집합** — 중복 없는 모음. `A not in B` (있나 없나 확인)를 빠르게 하려고 씀.
- `[c for c in df.columns if c not in non_desc]` : 'non_desc에 **없는** 열'만 골라 descriptor 열 목록으로.
- `FEATURES = { '이름': 표, ... }` : **딕셔너리**에 입력 3종을 '이름→데이터'로 담음. 아래 셀에서 하나씩 꺼내 반복 평가.

### 셀 3 — 채점 함수 정의 (혼동행렬 기반)
예측 확률을 0/1로 바꿔 TP/TN/FP/FN을 세고, 지표들을 계산해 돌려준다.

In [ ]:
# 성능 지표 함수 (논문 정의: 혼동행렬 TP/TN/FP/FN 기반)
def metrics_from(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    acc = (tp + tn) / (tp + tn + fp + fn)
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    mcc = matthews_corrcoef(y_true, pred)
    roc = roc_auc_score(y_true, proba)
    pr = average_precision_score(y_true, proba)
    return dict(ROC_AUC=roc, PR_AUC=pr, MCC=mcc, Accuracy=acc,
               Recall=rec, Precision=prec, TP=tp, TN=tn, FP=fp, FN=fn)

🔎 **코드 뜯어보기 (셀 3)**
- `def metrics_from(y_true, proba, thr=0.5):` : 함수 정의. `thr=0.5`는 **기본값** — 안 넘기면 0.5로 씀.
- `(proba >= thr)` : 확률이 0.5 이상인지 True/False 배열. `.astype(int)` : True/False를 **1/0**으로.
- `confusion_matrix(...).ravel()` : **혼동행렬**(맞고 틀림 표)을 만들고 `.ravel()`로 **네 값(tn,fp,fn,tp)** 을 한 줄로 펴서 한 번에 받음.
- `A if 조건 else B` : **조건부 표현** — 조건이 참이면 A, 아니면 B. 여기선 분모가 0일 때 나눗셈 오류를 피함.
- `matthews_corrcoef`, `roc_auc_score`, `average_precision_score` : 각각 **MCC, ROC-AUC, PR-AUC**를 계산하는 sklearn 함수.
- `return dict(이름=값, ...)` : 여러 결과를 **딕셔너리로 묶어** 한 번에 돌려줌.

### 셀 4 — 세 입력을 같은 방식으로 학습·채점
입력 3종을 반복하며 교차검증 예측→채점하고, 결과를 표로 저장한다.

In [ ]:
# 3종 특징 집합을 같은 조건(5-fold CV, RandomForest)으로 비교
def make_model():
    return RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                  random_state=42, n_jobs=-1)   # ← 괄호 안이 하이퍼파라미터(튜닝 대상)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows, proba_store = [], {}
for name, X in FEATURES.items():
    proba = cross_val_predict(make_model(), X, y, cv=skf,
                              method='predict_proba', n_jobs=-1)[:, 1]
    proba_store[name] = proba
    m = metrics_from(y, proba)
    m = {'features': name, **m}
    rows.append(m)
    print('[%s] ROC-AUC %.3f | PR-AUC %.3f | MCC %.3f | Acc %.3f | Recall %.3f | Prec %.3f'
          % (name, m['ROC_AUC'], m['PR_AUC'], m['MCC'], m['Accuracy'], m['Recall'], m['Precision']))

res = pd.DataFrame(rows)
OUT = 'data/HSD17B13_model_eval_1to1.csv'
res.to_csv(OUT, index=False)
print('\n결과 저장:', OUT)
print(res[['features', 'ROC_AUC', 'PR_AUC', 'MCC', 'Accuracy', 'Recall', 'Precision']]
      .round(3).to_string(index=False))

🔎 **코드 뜯어보기 (셀 4)**
- `def make_model(): return RandomForestClassifier(...)` : 매번 **새 모델**을 만들어 주는 함수. 괄호 안 값들이 **하이퍼파라미터**(사람이 정하는 설정): `n_estimators=300`(트리 300개), `class_weight='balanced'`(양·음 불균형 보정), `random_state=42`(재현 위해 난수 고정), `n_jobs=-1`(CPU 전부 사용).
- `StratifiedKFold(n_splits=5, shuffle=True, ...)` : 5조각으로 나눌 준비(shuffle=섞은 뒤 나눔).
- `for name, X in FEATURES.items():` : 딕셔너리의 **.items()** 로 '이름(name)과 값(X)'을 함께 반복.
- `cross_val_predict(모델, X, y, cv=skf, method='predict_proba')[:, 1]` : 교차검증으로 예측 확률을 구함. `[:, 1]` = 결과 표에서 **1번(=active)일 확률 열**만 선택.
- `m = {'features': name, **m}` : **`**m`** 은 딕셔너리를 '펼쳐서' 합치기 — 기존 지표에 'features' 항목을 더한 새 딕셔너리.
- `pd.DataFrame(rows)` : 딕셔너리들의 리스트를 **표**로. `.round(3)` : 소수 3자리로 반올림해 보기 좋게.

### 셀 5 — ROC · PR 곡선 그리기
입력 3종의 곡선을 한 그림에 겹쳐 그리고 PNG로 저장한다.

In [ ]:
# ROC / PR 커브 (3종 비교)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for name, proba in proba_store.items():
    fpr, tpr, _ = roc_curve(y, proba)
    ax[0].plot(fpr, tpr, label='%s (AUC=%.3f)' % (name, roc_auc_score(y, proba)))
    pr, rc, _ = precision_recall_curve(y, proba)
    ax[1].plot(rc, pr, label='%s (PR-AUC=%.3f)' % (name, average_precision_score(y, proba)))
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[0].set_title('ROC curve'); ax[0].legend(loc='lower right', fontsize=9)
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall curve'); ax[1].legend(loc='lower left', fontsize=9)
plt.tight_layout()
FIG = 'data/HSD17B13_model_eval_1to1.png'
plt.savefig(FIG, dpi=130)
plt.show()
print('그림 저장:', FIG)

🔎 **코드 뜯어보기 (셀 5)**
- `plt.subplots(1, 2, figsize=(12, 5))` : 그림 틀을 **1행 2칸**으로 만들고, `fig`(전체)와 `ax`(각 칸)를 받음. `ax[0]`=왼쪽, `ax[1]`=오른쪽.
- `roc_curve(y, proba)` / `precision_recall_curve(...)` : 곡선을 그릴 **좌표들**을 계산(반환값 중 안 쓸 것은 `_`로 받음).
- `ax[0].plot(x, y, label=...)` : 선 그리기. `label`은 범례에 표시할 이름.
- `'k--'` : 검은색(k) 점선(--). 대각선 = 무작위 기준.
- `.set_xlabel/.set_title/.legend` : 축 이름·제목·범례 설정.
- `plt.tight_layout()` : 겹침 없이 배치 정리. `plt.savefig(경로, dpi=130)` : 그림을 파일로 저장(dpi=해상도).
- `plt.show()` : 노트북에 그림 표시.